# Notebook 02 — Unificación de corpus

Adapta el **Spanish Hate Speech Superset** y **DETOXIS** al esquema canónico del proyecto y los concatena en `data/interim/corpus_combinado.parquet`.

**Referencia:** `guia.md` §7.2 Pipeline completo, §6.3 Esquema canónico.

> ⚠️ Ejecutar desde la raíz del proyecto (`Tesis_Proyecto/`).

## 0. Configuración de entorno

In [ ]:
import sys
from pathlib import Path

# Asegurar que la raíz del proyecto esté en el path
ROOT = Path().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f'Raíz del proyecto: {ROOT}')
print(f'Python: {sys.version}')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

from src.data.clean import normalizar
from src.data.unify import construir_corpus, RAW_SUPERSET, RAW_DETOXIS, OUT_INTERIM

print('Módulos importados correctamente.')

## 1. Inspección rápida de las fuentes

In [ ]:
# --- Superset ---
df_sup_raw = pd.read_csv(RAW_SUPERSET)
print('=== SUPERSET ===')
print(f'Shape: {df_sup_raw.shape}')
print(f'Columnas: {df_sup_raw.columns.tolist()}')
df_sup_raw.head(3)

In [ ]:
print('Distribución de etiquetas superset:')
print(df_sup_raw['labels'].value_counts())
print('\nPor dataset:')
print(df_sup_raw['dataset'].value_counts())

In [ ]:
# --- DETOXIS ---
df_det_raw = pd.read_csv(RAW_DETOXIS)
print('=== DETOXIS ===')
print(f'Shape: {df_det_raw.shape}')
print(f'Columnas: {df_det_raw.columns.tolist()}')
df_det_raw.head(3)

In [ ]:
print('Distribución de toxicity_level en DETOXIS:')
print(df_det_raw['toxicity_level'].value_counts().sort_index())
print(f'\nMapeo aplicado: toxicity_level >= 2 → hate (1)')
etiqueta_det = (df_det_raw['toxicity_level'] >= 2).astype(int)
print(etiqueta_det.value_counts())

## 2. Demo de normalización sobre muestras de DETOXIS

In [ ]:
# Verificar normalización en 5 ejemplos reales
muestras = df_det_raw['comment'].sample(5, random_state=42).tolist()
for texto in muestras:
    print(f'Original : {texto[:120]}')
    print(f'Limpio   : {normalizar(texto)[:120]}')
    print()

## 3. Construir corpus unificado

In [ ]:
corpus = construir_corpus(verbose=True)

In [ ]:
corpus.dtypes

In [ ]:
corpus.head(5)

## 4. Visualizaciones del corpus combinado

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Corpus Unificado — Visión general', fontsize=13, fontweight='bold')

# --- Distribución de clases global ---
labels_map = {0: 'No hate', 1: 'Hate'}
counts = corpus['etiqueta'].map(labels_map).value_counts()
axes[0].bar(counts.index, counts.values, color=['#4CAF50', '#F44336'])
axes[0].set_title('Distribución de clases')
axes[0].set_ylabel('Ejemplos')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 100, f'{v:,}\n({v/len(corpus):.1%})', ha='center', fontsize=9)

# --- Ejemplos por dataset ---
ds_counts = corpus['dataset'].value_counts()
axes[1].barh(ds_counts.index, ds_counts.values, color='#2196F3')
axes[1].set_title('Ejemplos por dataset')
axes[1].set_xlabel('Ejemplos')
for i, v in enumerate(ds_counts.values):
    axes[1].text(v + 30, i, str(v), va='center', fontsize=9)

# --- Distribución de longitudes ---
n_tokens = corpus['texto'].str.split().str.len()
axes[2].hist(n_tokens, bins=50, color='#9C27B0', edgecolor='white', alpha=0.8)
axes[2].axvline(n_tokens.median(), color='orange', linestyle='--', label=f'Mediana={n_tokens.median():.0f}')
axes[2].axvline(n_tokens.quantile(0.95), color='red', linestyle='--', label=f'P95={n_tokens.quantile(0.95):.0f}')
axes[2].set_title('Longitud de texto (tokens)')
axes[2].set_xlabel('Tokens')
axes[2].legend(fontsize=8)

plt.tight_layout()

# Guardar figura
fig_path = ROOT / 'data/reports_qc/figuras/corpus_unificado.png'
fig_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(fig_path, dpi=100, bbox_inches='tight')
print(f'Figura guardada en: {fig_path.relative_to(ROOT)}')
plt.show()

## 5. Verificar el Parquet generado

In [ ]:
# Leer de vuelta el Parquet para confirmar integridad
corpus_leido = pd.read_parquet(OUT_INTERIM)
print(f'Filas en Parquet: {len(corpus_leido):,}')
print(f'Columnas: {corpus_leido.columns.tolist()}')
print(f'\nTamaño en disco: {OUT_INTERIM.stat().st_size / 1024:.1f} KB')
assert len(corpus_leido) == len(corpus), 'El Parquet no coincide con el DataFrame en memoria'
print('\n✓ Parquet verificado correctamente.')

## 6. Resumen para EXPERIMENTOS.md

In [ ]:
hate = (corpus['etiqueta'] == 1).sum()
no_hate = (corpus['etiqueta'] == 0).sum()
n_tokens_col = corpus['texto'].str.split().str.len()

resumen = f"""
## Paso 1.5 — Corpus unificado interim

- Fecha: {pd.Timestamp.now().strftime('%Y-%m-%d')}
- Archivo: data/interim/corpus_combinado.parquet
- Total filas: {len(corpus):,}
- Hate (1): {hate:,} ({hate/len(corpus):.1%})
- No hate (0): {no_hate:,} ({no_hate/len(corpus):.1%})
- Mediana tokens: {n_tokens_col.median():.0f}
- P95 tokens: {n_tokens_col.quantile(0.95):.0f}
- Fuentes: {corpus['dataset'].nunique()} datasets
  - Superset: {(corpus['dataset'] != 'detoxis').sum():,} filas
  - DETOXIS: {(corpus['dataset'] == 'detoxis').sum():,} filas
"""
print(resumen)